# ARCHIVED: superseded by notebooks/01_generate_and_validate.ipynb
# SA Metric Branch Generation

Generate Structural Analysis branches for all metrics with **four fixed branch types**:

| Type | Role |
|------|------|
| **Bug** | Intentional defects in the target metric module only |
| **BugFX** | Same layout with defects fixed |
| **TCC** | Tool-optimized clean code + tool configs |
| **CC** | General clean code + smoke tests |

Branch pattern: `SA_<METRIC>_<TYPE>_<version>` (e.g. `SA_DOV_Bug_2.6`).

After generation, **four type-specific assert validators** confirm each branch was built correctly.

## Parameters

Edit the values below, then run the notebook top-to-bottom.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "scripts" / "sa_metrics.py").exists():
    ROOT = ROOT.parent  # running from notebooks/

SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))

from generate_sa_branches import generate_build
from validate_sa_branch import (
    TYPE_ASSERTERS,
    BranchValidationError,
    assert_bug_branch,
    assert_bugfx_branch,
    assert_tcc_branch,
    assert_cc_branch,
    validate_build,
)
from sa_metrics import FIXED_BRANCH_TYPES, METRIC_ABBREVS, branch_name, branches_for_metrics

print("Repo root:", ROOT)
print("Metrics:", ", ".join(METRIC_ABBREVS))
print("Fixed types:", ", ".join(FIXED_BRANCH_TYPES))

In [ ]:
VERSION = "2.6"       # branch suffix: SA_EPI_Bug_2.6
LANGUAGE = "python"   # written into sa/config.py
METRICS = "all"       # "all" or comma-separated: "EPI,DOV"
BUILD_DIR = "build"   # output folder under repo root
CREATE_GIT = False    # set True to create git branches after validation
PUSH_GIT = False      # requires CREATE_GIT=True

branch_list = branches_for_metrics(METRICS, VERSION)
print(f"Will generate {len(branch_list)} branches")
for i, name in enumerate(branch_list, 1):
    print(f"  {i:2d}. {name}")

## 1. Generate branches

In [ ]:
generated = generate_build(METRICS, VERSION, LANGUAGE, BUILD_DIR)
build_root = ROOT / BUILD_DIR
print(f"Generated {len(generated)} branches under {build_root}")

## 2. Assert validation (four branch types)

Each branch type has a dedicated validator:

| Assert function | Checks |
|-----------------|--------|
| `assert_bug_branch` | Defect markers in target module, partial tests, no TCC configs |
| `assert_bugfx_branch` | No defect markers, full tests, clean variant |
| `assert_tcc_branch` | `TOOL_MODE`, tool config files, tool-oriented tests |
| `assert_cc_branch` | CC-specific clean code, single smoke test, no tool configs |

Common checks (all types): folder name, `sa/config.py`, target module present, non-target modules are stubs.

In [ ]:
results = []
errors = []

for bname in generated:
    abbrev = bname.split("_")[1]
    btype = bname.split("_")[2]
    path = build_root / bname
    try:
        TYPE_ASSERTERS[btype](str(path), abbrev, VERSION, LANGUAGE)
        results.append((bname, btype, abbrev, "PASS", ""))
    except AssertionError as exc:
        errors.append((bname, str(exc)))
        results.append((bname, btype, abbrev, "FAIL", str(exc)))

print(f"{'Branch':<22} {'Type':<6} {'Metric':<6} Status")
print("-" * 50)
for bname, btype, abbrev, status, _ in results:
    mark = "OK" if status == "PASS" else "FAIL"
    print(f"{bname:<22} {btype:<6} {abbrev:<6} {mark}")

passed = sum(1 for r in results if r[3] == "PASS")
print(f"\n{passed}/{len(results)} branches passed")
if errors:
    print("\nFailures:")
    for bname, msg in errors:
        print(f"  {bname}: {msg}")
else:
    print("All branches created in the expected manner.")

### Per-type assert demo (one example each)

Run the four assert functions explicitly on sample branches for a clear pass/fail view.

In [ ]:
DEMO_METRIC = "DOV"  # change to inspect another metric

demos = [
    ("Bug", assert_bug_branch, branch_name(DEMO_METRIC, "Bug", VERSION)),
    ("BugFX", assert_bugfx_branch, branch_name(DEMO_METRIC, "BugFX", VERSION)),
    ("TCC", assert_tcc_branch, branch_name(DEMO_METRIC, "TCC", VERSION)),
    ("CC", assert_cc_branch, branch_name(DEMO_METRIC, "CC", VERSION)),
]

for label, fn, bname in demos:
    path = build_root / bname
    try:
        fn(str(path), DEMO_METRIC, VERSION, LANGUAGE)
        print(f"✓ assert_{label.lower()}_branch  →  {bname}")
    except AssertionError as exc:
        print(f"✗ assert_{label.lower()}_branch  →  {bname}: {exc}")

## 3. Optional — create git branches

Only run after all asserts pass.

In [ ]:
if errors:
    raise BranchValidationError("Fix validation failures before creating git branches")

if CREATE_GIT:
    from create_metric_branches import create_git_branches
    from generate_sa_branches import push_branches

    create_git_branches(METRICS, VERSION, LANGUAGE)
    if PUSH_GIT:
        push_branches(generated)
        print("Pushed branches to origin")
    else:
        print("Git branches created locally (not pushed)")
else:
    print("Skipped git branch creation (CREATE_GIT=False)")